In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    

developer = OpenAI()
validator = OpenAI()
auditor = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

developer_model = "gpt-5.4-nano"
validator_model = "gpt-5-nano"
auditor_model = "llama3.2"

OpenAI API Key exists and begins sk-proj-


In [6]:
developer_system = "You are a co-operative model developer who provides a Model Development document and \
    faces question from model validator and indirect questions from model auditors and tries to \
        answer the questions to the best of your knowledge based only on the document provided in the first prompt.\
            If you don't know the answer to something just say so. Answer concisely in sentences or less"

validator_system = "You are a highly inquisitive model validator. You question the model developer and \
    model development document and try to face\
    questions from the model auditor. If you're unsure about a question from the model auditor, you defer\
        it to the model developer.If you don't know the answer to something just say so. Ask very concise few points not exceeding 1 sentence. At maximum 1 question"

auditor_system = "You are a critical model auditor who questions the model validator and analyzes any points/questions\
    the vaidator may have missed or any gaps in the model development document. Ask very concise few points not exceeding 1 sentence. At maximum 1 question"

with open("Marketing_Model_Development_Document.txt", "r", encoding="utf-8") as f:
    MDD = f.read()


developer_messages = ["Hi there, I'm the model developer. \n \n For context, here is the Model Development Document (MDD) :\n\n"+MDD]
validator_messages = ["Hi, I'll analyze the MDD and ask questions to the model developer"]
auditor_messages = ["Hello guys, I'm the model auditor and I'll ask questions directly to the model validator\
    and questions from the MDD"]

In [7]:


def call_dev():
    messages = [{"role": "system", "content": developer_system}]
    for dev, val, aud in zip(developer_messages, validator_messages, auditor_messages):
        messages.append({"role": "assistant", "content": dev})
        messages.append({"role": "user", "content": f"Model Validator: {val}"})
        messages.append({"role": "user", "content": f"Model Auditor: {aud}"})

    response = developer.chat.completions.create(
        model=developer_model,
        messages=messages
    )
    return response.choices[0].message.content


def call_val():
    messages = [{"role": "system", "content": validator_system}]
    for dev, val, aud in zip(developer_messages, validator_messages, auditor_messages):
        messages.append({"role": "user", "content": f"Model Developer: {dev}"})
        messages.append({"role": "assistant", "content": val})
        messages.append({"role": "user", "content": f"Model Auditor: {aud}"})
    messages.append({"role": "user", "content": f"Developer: {developer_messages[-1]}"})
    response = validator.chat.completions.create(
        model=validator_model,
        messages=messages
    )
    return response.choices[0].message.content


def call_auditor():
    messages = [{"role": "system", "content": auditor_system}]
    for dev, val, aud  in zip(developer_messages, validator_messages, auditor_messages):
        messages.append({"role": "user", "content": f"Model Developer: {dev}"})
        messages.append({"role": "user", "content": f"Model Validator: {val}"})
        messages.append({"role": "assistant", "content": aud})
    messages.append({"role": "user","content": f"Developer: {developer_messages[-1]}"})
    messages.append({"role": "user","content": f"Validator: {validator_messages[-1]}"})
    response = auditor.chat.completions.create(
        model=auditor_model,
        messages=messages
    )
    return response.choices[0].message.content



In [8]:

display(Markdown(f"### Model Developer:\n{developer_messages[0][:35]}\n"))
display(Markdown(f"### Model Validator:\n{validator_messages[0]}\n"))
display(Markdown(f"### Model Auditor:\n{auditor_messages[0]}\n"))

for i in range(5):
    dev_next = call_dev()
    display(Markdown(f"### Model Developer: {i+1}:\n{dev_next}\n"))
    developer_messages.append(dev_next)

    val_next = call_val()
    display(Markdown(f"### Model Validator: {i+1} \n{val_next}\n"))
    validator_messages.append(val_next)

    aud_next = call_auditor()
    display(Markdown(f"### Model Auditor: {i+1}\n{aud_next}\n"))
    auditor_messages.append(aud_next)

### Model Developer:
Hi there, I'm the model developer. 


### Model Validator:
Hi, I'll analyze the MDD and ask questions to the model developer


### Model Auditor:
Hello guys, I'm the model auditor and I'll ask questions directly to the model validator    and questions from the MDD


### Model Developer: 1:
Understood. I’ll answer questions only based on the provided MDD; if something isn’t specified there, I’ll say I don’t know.

Go ahead with your first question(s).


### Model Validator: 1 
Please specify the calibration method used when adjusting predicted probabilities (e.g., Platt scaling, isotonic regression, or another approach) and define the exact threshold for 'material divergence' that triggers recalibration.


...

### Model Validator: 4 
Is there any internal guideline or supplementary documentation beyond the MDD that requires investigators to make a quantitative judgment (magnitude) of divergence, rather than solely flagging issues?


### Model Auditor: 4
Validator: Is there an internal guideline, such as an AML (Anti-Money Laundering) or Credit Risk Policy manual, that might dictate a specific calibration error threshold or PSI delta for triggering recalibration of the model, although not explicitly stated in the MDD?


### Model Developer: 5:
I don’t know—there’s no supplementary or internal guideline documented beyond the MDD in the information provided.

The MDD itself does not mention any specific policy/manual or any required quantitative divergence thresholds for deciding when to recalibrate.


### Model Validator: 5 
There is no documented internal guideline beyond the MDD in the provided materials, and no evidence of a policy dictating a specific calibration threshold.


### Model Auditor: 5
We've exhausted the information provided in the MDD and the conversation with the model developer.
